# Schema Detection: Heuristic vs RAG-LLM

This notebook compares the existing heuristic schema change detector against an LLM-based detector that uses the same machine-readable schemas as non-parametric context.
It focuses on the canonical ↔ V2 scenario so that schema detection is evaluated on the same mapping used for canonical ingestion in the rest of the thesis.


In [1]:
import os

# Ensure the Hugging Face router environment is configured before running LLM-based detection.
# In a terminal (or your IDE), set for example:
#   export HF_API_TOKEN="<your_hf_token>"
#   export HF_CHAT_URL="https://router.huggingface.co/v1/chat/completions"
# The notebook then reuses these values from the environment.

if "HF_CHAT_URL" not in os.environ:
    os.environ["HF_CHAT_URL"] = "https://router.huggingface.co/v1/chat/completions"

In [2]:
from pathlib import Path
import json
import subprocess
import pandas as pd

# Resolve project root from this notebook's working directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCHEMAS_DIR = PROJECT_ROOT / "schemas"
DOCS_DIR = PROJECT_ROOT / "docs"
RESULTS_DIR = PROJECT_ROOT / "results"

# We evaluate schema detection for the canonical ↔ V2 scenario used in canonical ingestion.
GROUND_TRUTH_PATH = RESULTS_DIR / "canonical_changes_v2.json"

PROJECT_ROOT, SCHEMAS_DIR, RESULTS_DIR

(PosixPath('/Users/rajakarthikchirumamilla/Documents/ThesisWork/etl-ai-schema'),
 PosixPath('/Users/rajakarthikchirumamilla/Documents/ThesisWork/etl-ai-schema/schemas'),
 PosixPath('/Users/rajakarthikchirumamilla/Documents/ThesisWork/etl-ai-schema/results'))

In [3]:
# Ensure canonical ground-truth change set exists
if not GROUND_TRUTH_PATH.exists():
    print("canonical_changes_v2.json not found in results/.")
    print(
        "Run: python scripts/evaluate_baseline_vs_llm_canonical.py --version v2 --model llama3 --output results/canonical_eval_llama3.json"
    )
    raise FileNotFoundError(GROUND_TRUTH_PATH)
else:
    print("Canonical ground-truth change set found:", GROUND_TRUTH_PATH)

# Configure which LLM model to use for schema detection
# You can change this to "llama3", "mistral", or keep the HF model below.
MODEL = "hf:openai/gpt-oss-120b:groq"

def _safe_label(label: str) -> str:
    cleaned = label.replace("hf:", "HF_").replace("hf/", "HF_")
    cleaned = cleaned.replace("/", "_").replace(":", "_")
    return cleaned

label = _safe_label(MODEL)
llm_changes_path = RESULTS_DIR / f"canonical_changes_v2_llm_{label}.json"

if not llm_changes_path.exists():
    print(f"LLM changes JSON not found for model={MODEL}, running canonical LLM-based detector ...")
    subprocess.run(
        ["python", "scripts/run_canonical_schema_detection_llm.py", "--model", MODEL],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print("LLM changes JSON already present:", llm_changes_path)

GROUND_TRUTH_PATH, llm_changes_path

Canonical ground-truth change set found: /Users/rajakarthikchirumamilla/Documents/ThesisWork/etl-ai-schema/results/canonical_changes_v2.json
LLM changes JSON not found for model=hf:openai/gpt-oss-120b:groq, running canonical LLM-based detector ...
Running LLM-based schema detection for canonical ↔ V2 using model 'hf:openai/gpt-oss-120b:groq'...
✓ Saved LLM-inferred canonical↔V2 changes to /Users/rajakarthikchirumamilla/Documents/ThesisWork/etl-ai-schema/results/canonical_changes_v2_llm_HF_openai_gpt-oss-120b_groq.json


(PosixPath('/Users/rajakarthikchirumamilla/Documents/ThesisWork/etl-ai-schema/results/canonical_changes_v2.json'),
 PosixPath('/Users/rajakarthikchirumamilla/Documents/ThesisWork/etl-ai-schema/results/canonical_changes_v2_llm_HF_openai_gpt-oss-120b_groq.json'))

In [4]:
# Load ground truth and detection outputs and compute precision/recall/F1

# Ground truth and heuristic baseline: canonical_changes_v2.json
with GROUND_TRUTH_PATH.open("r") as f:
    gt_changes = json.load(f)

gt_added = {c["column"] for c in gt_changes.get("added_columns", [])}
gt_removed = {c["column"] for c in gt_changes.get("removed_columns", [])}
gt_renamed = {
    (c["old_column"], c["new_column"])
    for c in gt_changes.get("renamed_columns", [])
}

def _load_changes(path: Path):
    with path.open("r") as f:
        return json.load(f)

# Heuristic detector defines the ground truth here
heuristic_changes = gt_changes
llm_changes = _load_changes(llm_changes_path)

def prf(tp: int, fp: int, fn: int):
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return prec, rec, f1

def eval_detector(det_changes: dict):
    # Normalise structure to the expected keys
    det = {
        "added_columns": det_changes.get("added_columns", []),
        "removed_columns": det_changes.get("removed_columns", []),
        "renamed_columns": det_changes.get("renamed_columns", []),
    }

    det_renamed = {
        (r.get("old_column"), r.get("new_column"))
        for r in det["renamed_columns"]
        if r.get("old_column") and r.get("new_column")
    }

    det_added = {c["column"] for c in det["added_columns"] if "column" in c}
    det_removed = {c["column"] for c in det["removed_columns"] if "column" in c}

    # Exclude endpoints of renames from added/removed, like in the evaluation script
    renamed_old = {old for (old, _new) in det_renamed}
    renamed_new = {_new for (old, _new) in det_renamed}
    det_added -= renamed_new
    det_removed -= renamed_old

    # Added
    tp_added = len(det_added & gt_added)
    fp_added = len(det_added - gt_added)
    fn_added = len(gt_added - det_added)
    p_add, r_add, f_add = prf(tp_added, fp_added, fn_added)

    # Removed
    tp_removed = len(det_removed & gt_removed)
    fp_removed = len(det_removed - gt_removed)
    fn_removed = len(gt_removed - det_removed)
    p_rem, r_rem, f_rem = prf(tp_removed, fp_removed, fn_removed)

    # Renamed
    tp_ren = len(det_renamed & gt_renamed)
    fp_ren = len(det_renamed - gt_renamed)
    fn_ren = len(gt_renamed - det_renamed)
    p_ren, r_ren, f_ren = prf(tp_ren, fp_ren, fn_ren)

    # Overall (added + removed)
    tp_all = tp_added + tp_removed
    fp_all = fp_added + fp_removed
    fn_all = fn_added + fn_removed
    p_all, r_all, f_all = prf(tp_all, fp_all, fn_all)

    return {
        "added": {"precision": p_add, "recall": r_add, "f1": f_add},
        "removed": {"precision": p_rem, "recall": r_rem, "f1": f_rem},
        "renamed": {"precision": p_ren, "recall": r_ren, "f1": f_ren},
        "overall": {"precision": p_all, "recall": r_all, "f1": f_all},
    }

metrics_heuristic = eval_detector(heuristic_changes)
metrics_llm = eval_detector(llm_changes)

metrics_heuristic, metrics_llm

({'added': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
  'removed': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
  'renamed': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
  'overall': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}},
 {'added': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
  'removed': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
  'renamed': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0},
  'overall': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}})

In [5]:
# Build a comparison table for the thesis

rows = []
for method_name, metrics in [("Heuristic", metrics_heuristic), ("RAG-LLM", metrics_llm)]:
    for change_type in ["added", "removed", "renamed", "overall"]:
        m = metrics[change_type]
        rows.append(
            {
                "Method": method_name,
                "Change type": change_type,
                "Precision": round(m["precision"], 2),
                "Recall": round(m["recall"], 2),
                "F1": round(m["f1"], 2),
            }
        )

df_metrics = pd.DataFrame(rows)
df_metrics

,Method,Change type,Precision,Recall,F1
0,Heuristic,added,1.0,1.0,1.0
1,Heuristic,removed,1.0,1.0,1.0
2,Heuristic,renamed,1.0,1.0,1.0
3,Heuristic,overall,1.0,1.0,1.0
4,RAG-LLM,added,1.0,1.0,1.0
5,RAG-LLM,removed,1.0,1.0,1.0
6,RAG-LLM,renamed,1.0,1.0,1.0
7,RAG-LLM,overall,1.0,1.0,1.0
